Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
A function or coroutine to execute.

In [2]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")


d:\ai-engineer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
response = model.invoke("Write me a 200 words paragraph on Artificial Intelligence")
response.content

"Artificial Intelligence (AI) refers to the simulation of human intelligence processes by machines, particularly computer systems. It encompasses a broad range of capabilities, allowing machines to learn from experience, adapt to new inputs, perform human-like reasoning, and execute tasks traditionally requiring human cognitive functions. Key areas include machine learning, natural language processing, computer vision, and robotics.\n\nAI's transformative power is evident across numerous sectors. In healthcare, it aids in diagnostics, drug discovery, and personalized treatment plans. Finance leverages AI for fraud detection, algorithmic trading, and risk assessment. From optimizing supply chains and powering autonomous vehicles to enhancing customer service through chatbots and personal assistants, AI is fundamentally reshaping industries and daily life. It enables sophisticated data analysis, automates complex processes, and unlocks unprecedented efficiencies.\n\nWhile offering immens

In [6]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""
    # Here you would implement the logic to fetch weather data from an API
    return f"The current weather in {location} is sunny with a temperature of 25°C."



model_with_tools=model.bind_tools(get_weather)


convert_to_genai_function_declarations expects a Sequence and not a single tool.


In [7]:

response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")


Key 'additional_properties' is not supported in schema, ignoring
Key 'defs' is not supported in schema, ignoring
Key 'ref' is not supported in schema, ignoring
Key 'any_of' is not supported in schema, ignoring
Key 'example' is not supported in schema, ignoring
Key 'max_items' is not supported in schema, ignoring
Key 'max_length' is not supported in schema, ignoring
Key 'max_properties' is not supported in schema, ignoring
Key 'min_items' is not supported in schema, ignoring
Key 'min_length' is not supported in schema, ignoring
Key 'min_properties' is not supported in schema, ignoring
Key 'property_ordering' is not supported in schema, ignoring


content='' additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}, '__gemini_function_call_thought_signatures__': {'80afa81e-1b35-42a4-875f-1e5468b2fee6': 'CukBARFNMg9c1Piq1LjEPcAXzRLBTQe/R4RA/M3oxt18F8jAUtcDvSgY704tv3LYZ9NLvIKaPD/DbcuoiBXmCxhOj564gIeRCTv5whKPtZkHGp57Wq9o02nKyqes4vNi+rHVFxAK/OzmL+tUw/ft0jD/W2etzPuYG1LfrnJBVMa+1KF/bJx7dZZWUUcrsbrghCRZajIwY8DpJFo0NoFKH57SB3PvYMjdX2DM0IQwPYaKGTry8MszIx5lwhYCs50anHViDymRRO4WCZfTneZ89N1lyj+FZneQ+wZjY5Qwbs/NtKPoQkTRGzigmUA='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019fea35-2297-7cf1-9e92-786c5ce66d9c-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '80afa81e-1b35-42a4-875f-1e5468b2fee6', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 51, 'output_tokens': 61, 'total_tokens': 112, 'input_token_details': {'cache_read': 0}, 'output_token_de

Tool Execution Loop

In [8]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

Key 'additional_properties' is not supported in schema, ignoring
Key 'defs' is not supported in schema, ignoring
Key 'ref' is not supported in schema, ignoring
Key 'any_of' is not supported in schema, ignoring
Key 'example' is not supported in schema, ignoring
Key 'max_items' is not supported in schema, ignoring
Key 'max_length' is not supported in schema, ignoring
Key 'max_properties' is not supported in schema, ignoring
Key 'min_items' is not supported in schema, ignoring
Key 'min_length' is not supported in schema, ignoring
Key 'min_properties' is not supported in schema, ignoring
Key 'property_ordering' is not supported in schema, ignoring
Key 'additional_properties' is not supported in schema, ignoring
Key 'defs' is not supported in schema, ignoring
Key 'ref' is not supported in schema, ignoring
Key 'any_of' is not supported in schema, ignoring
Key 'example' is not supported in schema, ignoring
Key 'max_items' is not supported in schema, ignoring
Key 'max_length' is not supported 

The current weather in Boston is sunny with a temperature of 25°C.


In [9]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}, '__gemini_function_call_thought_signatures__': {'138b6452-d48e-4277-93cd-4403a86adc33': 'CukBARFNMg9u9dib1JQ1digfm/8einYmMHvOXRoHO0FFHJlWhmhr6lYZYZt8MtU2gh/HvONX1rd3W0WJMs/p7mxMjSLc7ov9P/EWK6tQJzfb1v5iiNempWKuIK/RoP1UuksC0m/dVL1cZw1Dqb0VyEXzZ7f0IXMhG94pTUcfm5c5BpZAWwNK2UhEeOTXqiGPkfRat5DMXjexPxKhm71Ek4sRzCtw1VI48FzwBGMa6JOm7DV9zPce38h4A12MyYxaAk89o48jtz03b8Q6G5kqHsvZPUoQtPfjmG4Dg8iG6GMuyaplAx7+msDGL2g='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fea3f-c0de-7b21-8974-e6231cb9b6ee-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '138b6452-d48e-4277-93cd-4403a86adc33', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 50, 'output_tokens': 61,